# EV-Insight 机器学习建模
---
本 Notebook 实现论文第 3.2 节的建模全流程。

**执行步骤:**
1. K-Means 聚类 - 车型细分市场识别
2. 随机森林回归 - 价格预测模型
3. 梯度提升回归 - 对比模型
4. 线性回归 - 基线模型
5. 特征重要性分析
6. 模型保存

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_percentage_error, silhouette_score

print('✅ 依赖库加载完成')

In [ ]:
# 加载清洗后的数据
df = pd.read_csv('../data/ev_data_cleaned.csv')
print(f'数据: {len(df)} 条记录, {len(df.columns)} 个特征')
print(f'品牌数: {df["brand"].nunique()}')
print(f'价格区间: {df["price_median"].min():.1f} ~ {df["price_median"].max():.1f} 万')

## 1. K-Means 车型聚类

In [ ]:
# 聚类特征
cluster_features = ['price_median', 'range_km', 'battery_capacity', 
                    'power_kw', 'length_mm', 'adas_level_num', 'accel_0_100']
cluster_features = [c for c in cluster_features if c in df.columns]
print(f'聚类特征: {cluster_features}')

# 准备数据
X_cluster = df[cluster_features].fillna(df[cluster_features].median())
scaler_cluster = StandardScaler()
X_scaled = scaler_cluster.fit_transform(X_cluster)

In [ ]:
# 肘部法则：寻找最优 K
inertias = []
silhouettes = []
K_range = range(2, 8)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = kmeans.fit_predict(X_scaled)
    inertias.append(kmeans.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(K_range, inertias, 'o-', color='#667eea', linewidth=2, markersize=8)
axes[0].set_xlabel('聚类数 K')
axes[0].set_ylabel('惯性 (Inertia)')
axes[0].set_title('肘部法则 (Elbow Method)')
axes[0].axvline(x=3, color='red', linestyle='--', alpha=0.5, label='K=3')
axes[0].legend()

axes[1].plot(K_range, silhouettes, 'o-', color='#22c55e', linewidth=2, markersize=8)
axes[1].set_xlabel('聚类数 K')
axes[1].set_ylabel('轮廓系数 (Silhouette Score)')
axes[1].set_title('轮廓系数法')
axes[1].axvline(x=3, color='red', linestyle='--', alpha=0.5, label='K=3')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'K=3 轮廓系数: {silhouettes[1]:.4f}')

In [ ]:
# 训练最终 K-Means (K=3)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=20, max_iter=500)
df['cluster'] = kmeans.fit_predict(X_scaled)

# 各类别统计
for c in range(3):
    cluster_data = df[df['cluster'] == c]
    print(f'\n聚类 {c}:')
    print(f'  车型数: {len(cluster_data)} ({len(cluster_data)/len(df)*100:.1f}%)')
    print(f'  平均价格: {cluster_data["price_median"].mean():.1f} 万')
    print(f'  平均续航: {cluster_data["range_km"].mean():.0f} km')
    print(f'  代表品牌: {cluster_data["brand"].value_counts().head(3).index.tolist()}')

In [ ]:
# PCA 降维可视化
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
df['pca_x'] = X_pca[:, 0]
df['pca_y'] = X_pca[:, 1]

colors = ['#22c55e', '#3b82f6', '#ef4444']
labels_map = {0: '经济代步型', 1: '中端家用型', 2: '高端性能型'}

fig, ax = plt.subplots(figsize=(10, 7))
for c in range(3):
    mask = df['cluster'] == c
    ax.scatter(df.loc[mask, 'pca_x'], df.loc[mask, 'pca_y'],
               c=colors[c], label=labels_map[c], alpha=0.6, s=10)

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} 方差)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} 方差)')
ax.set_title('K-Means 聚类结果 (PCA 2D 投影)')
ax.legend()
plt.tight_layout()
plt.show()

## 2. 价格预测模型

In [ ]:
# 准备特征
feature_cols = [
    'battery_capacity', 'range_km', 'power_kw', 'torque_nm', 'accel_0_100',
    'length_mm', 'width_mm', 'height_mm', 'wheelbase_mm', 'adas_level_num',
    'seats', 'range_price_ratio', 'brand_premium_index', 'power_score',
    'space_index', 'tech_score', 'brand_encoded', 'battery_type_encoded',
    'body_type_encoded', 'energy_type_encoded'
]
feature_cols = [c for c in feature_cols if c in df.columns]
print(f'使用 {len(feature_cols)} 个特征')

X = df[feature_cols].fillna(df[feature_cols].median())
y = df['price_median'].fillna(df['price_median'].median())

# 划分数据集
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f'训练集: {len(X_train)}, 测试集: {len(X_test)}')

# 标准化
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# 训练三种模型并对比
models = {
    '线性回归': LinearRegression(),
    '随机森林': RandomForestRegressor(n_estimators=200, max_depth=20, 
                                       random_state=42, n_jobs=-1),
    '梯度提升': GradientBoostingRegressor(n_estimators=200, max_depth=5,
                                          learning_rate=0.1, random_state=42),
}

results = []
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test, y_pred) * 100
    results.append({'模型': name, 'RMSE(万)': rmse, 'R²': r2, 'MAPE(%)': mape})
    print(f'{name}: RMSE={rmse:.2f}万, R²={r2:.3f}, MAPE={mape:.1f}%')

results_df = pd.DataFrame(results)
results_df

In [ ]:
# 预测效果可视化
rf_model = models['随机森林']
y_pred_rf = rf_model.predict(X_test_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 预测 vs 实际散点图
axes[0].scatter(y_test, y_pred_rf, alpha=0.4, s=15, color='#667eea')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 
             'r--', linewidth=2, label='完美预测')
axes[0].set_xlabel('实际价格 (万元)')
axes[0].set_ylabel('预测价格 (万元)')
axes[0].set_title(f'随机森林预测效果 (R²={r2_score(y_test, y_pred_rf):.3f})')
axes[0].legend()

# 残差分布
residuals = y_test - y_pred_rf
axes[1].hist(residuals, bins=40, color='#22c55e', edgecolor='white', alpha=0.8)
axes[1].axvline(0, color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('残差 (万元)')
axes[1].set_ylabel('频数')
axes[1].set_title(f'残差分布 (标准差={residuals.std():.2f}万)')

plt.tight_layout()
plt.show()

In [ ]:
# 特征重要性分析
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
top_features = feature_importance.head(10)
ax.barh(top_features['feature'][::-1], top_features['importance'][::-1], color='#667eea')
ax.set_xlabel('重要性得分')
ax.set_title('随机森林特征重要性 (Top 10)')
plt.tight_layout()
plt.show()

feature_importance.head(10)

In [ ]:
# 保存模型
joblib.dump(rf_model, '../app/rf_price_model.pkl')
joblib.dump(scaler, '../app/scaler.pkl')
joblib.dump(feature_cols, '../app/feature_names.pkl')
joblib.dump(kmeans, '../app/kmeans_model.pkl')
joblib.dump(scaler_cluster, '../app/cluster_scaler.pkl')

print('✅ 所有模型已保存到 ../app/ 目录')
print('  - rf_price_model.pkl   (随机森林回归)')
print('  - scaler.pkl           (特征标准化)')
print('  - feature_names.pkl    (特征名列表)')
print('  - kmeans_model.pkl     (K-Means 聚类)')
print('  - cluster_scaler.pkl   (聚类标准化)')